In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import time 

from vitpol import ViT

def print_p_values(model, log_file=None):
    print("\n=== FINAL P VALUES ===")
    for name, module in model.named_modules():
        if hasattr(module, "p_raw"):
            p = F.softplus(module.p_raw).detach().cpu()
            output = f"{name}: {p.numpy()}"
            print(output)
            if log_file is not None:
                log_file.write(output + "\n")

def main():
    model = ViT(img_size=32, patch_size=8, in_channels=3, num_classes=10)
    batch_size = 64
    epochs = 50
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # 📦 Данные
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # --- Функции обучения ---
    def train_epoch(model, loader):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    def test_epoch(model, loader):
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    # 📝 Логирование и тайминг
    log_file = open("training_log.txt", "w", encoding="utf-8")
    p_log = open("p_log.txt", "w", encoding="utf-8")
    best_test_loss = float('inf')
    
    epoch_times = [] # Список для хранения времени каждой эпохи

    print(f"Starting training on {device}...")
    
    for epoch in range(epochs):
        # 1. Замеряем начало
        if device.type == 'cuda': torch.cuda.synchronize()
        start_time = time.time()

        train_loss, train_acc = train_epoch(model, train_loader)
        test_loss, test_acc = test_epoch(model, test_loader)
        scheduler.step()

        # 2. Замеряем конец
        if device.type == 'cuda': torch.cuda.synchronize()
        duration = time.time() - start_time
        epoch_times.append(duration) # Сохраняем результат

        status = (f"Epoch {epoch+1:02d}/{epochs} | "
                  f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | "
                  f"Time: {duration:.2f}s")
        print(status)

        log_file.write(status + "\n")
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            torch.save(model.state_dict(), 'best_vit.pth')

    # --- Итоговая статистика ---
    avg_time = sum(epoch_times) / len(epoch_times)
    total_time = sum(epoch_times)

    print("\n" + "="*40)
    print(f"Total training time: {total_time/60:.2f} min")
    print(f"Average time per epoch: {avg_time:.2f} s")
    print_p_values(model, p_log)
    print("="*40)

    log_file.close()
    p_log.close()

if __name__ == "__main__":
    main()

Files already downloaded and verified
Files already downloaded and verified
Starting training on cuda...
Epoch 01/50 | Train Acc: 0.3830 | Test Acc: 0.4684 | Time: 32.84s
Epoch 02/50 | Train Acc: 0.4787 | Test Acc: 0.5133 | Time: 27.18s
Epoch 03/50 | Train Acc: 0.5210 | Test Acc: 0.5352 | Time: 26.72s
Epoch 04/50 | Train Acc: 0.5464 | Test Acc: 0.5565 | Time: 28.14s
Epoch 05/50 | Train Acc: 0.5677 | Test Acc: 0.5745 | Time: 25.98s
Epoch 06/50 | Train Acc: 0.5845 | Test Acc: 0.5848 | Time: 25.72s
Epoch 07/50 | Train Acc: 0.5994 | Test Acc: 0.5883 | Time: 24.91s
Epoch 08/50 | Train Acc: 0.6089 | Test Acc: 0.5961 | Time: 24.38s
Epoch 09/50 | Train Acc: 0.6206 | Test Acc: 0.6048 | Time: 24.60s
Epoch 10/50 | Train Acc: 0.6373 | Test Acc: 0.6052 | Time: 24.77s
Epoch 11/50 | Train Acc: 0.6464 | Test Acc: 0.6170 | Time: 24.61s
Epoch 12/50 | Train Acc: 0.6574 | Test Acc: 0.6233 | Time: 24.36s
Epoch 13/50 | Train Acc: 0.6676 | Test Acc: 0.6193 | Time: 24.36s
Epoch 14/50 | Train Acc: 0.6778 | Tes